# Data Preprocessing: AVGO 1-Minute Stock Data

Raw data: `14d_AVGO_1min.csv` — 7935 rows of 1-min OHLCV data

**Steps:**
1. Parse timestamps, handle gaps
2. Feature engineering (returns, volatility, volume features)
3. Outlier handling
4. Normalization
5. Train/val/test split

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, RobustScaler

sns.set_style('darkgrid')
%matplotlib inline

print('Libraries loaded.')

## 1. Load & Inspect

In [ ]:
df = pd.read_csv('14d_AVGO_1min.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

print(f'Shape: {df.shape}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'Unique trading days: {df.index.date}.nunique()')
df.head()

In [ ]:
# Check basic stats & missing
print('Missing values:\n', df.isnull().sum())
print('\nDuplicated timestamps:', df.index.duplicated().sum())
print('\nDescribe:')
df.describe()

## 2. Resample to 1-min Grid & Fill Gaps

Data only covers trading hours (4:00–19:59 ET). We resample to a continuous 1-min grid per day and forward-fill short gaps (≤5 min). Longer gaps (overnight/weekend) are left as NaN.

In [ ]:
# Create complete 1-min index per trading day
trading_days = sorted(set(df.index.date))
full_idx = pd.DatetimeIndex([])
for d in trading_days:
    day_start = pd.Timestamp(d).replace(hour=4, minute=0)
    day_end = pd.Timestamp(d).replace(hour=19, minute=59)
    day_range = pd.date_range(start=day_start, end=day_end, freq='1min')
    full_idx = full_idx.append(day_range)

# Reindex & fill short gaps only
df_full = df.reindex(full_idx)

# Forward-fill gaps ≤ 5 min, leave longer gaps as NaN
gap_mask = df_full['close'].isna()
gap_groups = (gap_mask != gap_mask.shift()).cumsum()
gap_lengths = gap_groups.map(gap_groups.value_counts())

short_gaps = gap_mask & (gap_lengths <= 5)
df_full[short_gaps] = df_full[short_gaps].ffill()

print(f'After resample: {df_full.shape[0]} rows')
print(f'Remaining NaN: {df_full.isnull().sum().sum()}')

## 3. Feature Engineering

In [ ]:
# Price-based features
df_full['returns'] = df_full['close'].pct_change()
df_full['log_returns'] = np.log(df_full['close'] / df_full['close'].shift(1))
df_full['price_range'] = df_full['high'] - df_full['low']
df_full['hl_pct'] = df_full['price_range'] / df_full['close'] * 100
df_full['mid_price'] = (df_full['high'] + df_full['low']) / 2

# Volume features
df_full['volume_change'] = df_full['volume'].pct_change()
df_full['log_volume'] = np.log1p(df_full['volume'])
df_full['dollar_volume'] = df_full['close'] * df_full['volume']

# Rolling features (5, 15, 30 min windows)
for w in [5, 15, 30]:
    df_full[f'sma_{w}'] = df_full['close'].rolling(w).mean()
    df_full[f'volatility_{w}'] = df_full['returns'].rolling(w).std()
    df_full[f'volume_sma_{w}'] = df_full['volume'].rolling(w).mean()
    df_full[f'high_{w}'] = df_full['high'].rolling(w).max()
    df_full[f'low_{w}'] = df_full['low'].rolling(w).min()

# Price position within rolling range
for w in [5, 15, 30]:
    roll_high = df_full['high'].rolling(w).max()
    roll_low = df_full['low'].rolling(w).min()
    df_full[f'pos_{w}'] = (df_full['close'] - roll_low) / (roll_high - roll_low + 1e-10)

print(f'Features engineered. Shape: {df_full.shape}')
df_full.head()

## 4. Handle Outliers (Winsorize)

In [ ]:
def winsorize_series(s, limits=(0.01, 0.01)):
    """Clip outliers at given percentiles."""
    lo, hi = s.quantile(limits[0]), s.quantile(1 - limits[1])
    return s.clip(lo, hi)

cols_to_winsorize = ['returns', 'log_returns', 'price_range', 'hl_pct',
                     'volume_change', 'dollar_volume']

for c in cols_to_winsorize:
    if c in df_full.columns:
        df_full[c] = winsorize_series(df_full[c])

print('Outliers winsorized at 1st/99th percentiles.')

## 5. Drop NaN Rows (from rolling windows)

In [ ]:
before = df_full.shape[0]
df_clean = df_full.dropna()
after = df_clean.shape[0]
print(f'Dropped {before - after} NaN rows (rolling window startup).')
print(f'Cleaned shape: {df_clean.shape}')

## 6. Normalize Features

In [ ]:
# Define feature groups
level_cols = ['open', 'high', 'low', 'close', 'mid_price']
price_feat = ['returns', 'log_returns', 'price_range', 'hl_pct']
vol_feat = ['log_volume', 'volume_change', 'dollar_volume']
roll_feat = [c for c in df_clean.columns if any(f in c for f in ['sma_', 'volatility_', 'volume_sma_', 'high_5', 'high_15', 'high_30', 'low_5', 'low_15', 'low_30', 'pos_'])]

# StandardScaler for returns-like; RobustScaler for price levels (has outliers)
standard_scaler = StandardScaler()
robust_scaler = RobustScaler()

# Fit on training portion (first 70% chronological)
split_idx = int(len(df_clean) * 0.7)

df_norm = df_clean.copy()

# Normalize price levels with RobustScaler
df_norm[level_cols] = robust_scaler.fit_transform(df_clean[level_cols])

# Normalize other features with StandardScaler
all_feat = price_feat + vol_feat + roll_feat
all_feat = [c for c in all_feat if c in df_clean.columns]
df_norm[all_feat] = standard_scaler.fit_transform(df_clean[all_feat])

print('Feature normalization complete.')
print(f'Level columns: {level_cols}\nFeature columns: {len(all_feat)}')

## 7. Train / Val / Test Split (Chronological)

In [ ]:
n = len(df_norm)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

train = df_norm.iloc[:train_end].copy()
val = df_norm.iloc[train_end:val_end].copy()
test = df_norm.iloc[val_end:].copy()

print(f'Train:  {train.shape[0]:>6} rows  ({train.index[0].date()} → {train.index[-1].date()})')
print(f'Val:    {val.shape[0]:>6} rows  ({val.index[0].date()} → {val.index[-1].date()})')
print(f'Test:   {test.shape[0]:>6} rows  ({test.index[0].date()} → {test.index[-1].date()})')

## 8. Save Preprocessed Data

In [ ]:
# Save full preprocessed data
df_norm.to_csv('avgo_preprocessed.csv')
train.to_csv('avgo_train.csv')
val.to_csv('avgo_val.csv')
test.to_csv('avgo_test.csv')

print('Saved: avgo_preprocessed.csv, avgo_train.csv, avgo_val.csv, avgo_test.csv')

## 9. Quick Visual Check

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Close price
axes[0].plot(train.index, train['close'], label='Train', alpha=0.7)
axes[0].plot(val.index, val['close'], label='Val', alpha=0.7)
axes[0].plot(test.index, test['close'], label='Test', alpha=0.7)
axes[0].set_title('Normalized Close Price')
axes[0].legend()

# Returns
axes[1].plot(df_norm.index, df_norm['returns'], alpha=0.5, color='gray')
axes[1].set_title('Normalized Returns (winsorized)')

# Volume
axes[2].plot(df_norm.index, df_norm['log_volume'], alpha=0.5, color='green')
axes[2].set_title('Normalized Log Volume')

plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print('='*60)
print('PREPROCESSING SUMMARY')
print('='*60)
print(f'Original rows: {len(df)}')
print(f'After resample: {len(df_full)}')
print(f'After dropna:   {len(df_norm)}')
print(f'Features:       {df_norm.shape[1]}')
print(f'Train:Val:Test  = {len(train)}:{len(val)}:{len(test)}')
print(f'Scaling:        RobustScaler(levels), StandardScaler(features)')
print(f'Outliers:       Winsorized at 1%/99%')
print('='*60)